In [127]:
import json
import os
from typing import Dict, Any, List, Optional
import pandas as pd
from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go

default_sample = """\
{"timestamp": 1759790081.822003, "device": "phone_2", "action": "idle", "time": 0.01679604311979629}
{"timestamp": 1759790081.822016, "time": 0.0799571066350519, "action": "transmit_end", "id": "all_gather_reduce_from_model_parallel_region_phone_1_phone_2_0", "internal_id": 1363, "duration": 0.07533652252252253}
{"timestamp": 1759790081.822017, "device": "phone_2", "action": "running", "time": 0.0799571066350519}
{"timestamp": 1759790081.846875, "time": 0.10479856463331555, "action": "transmit_start", "id": "all_gather_reduce_from_model_parallel_region_phone_2_phone_1_0", "internal_id": 1364, "size": 8286208.0}
{"timestamp": 1759790081.846889, "device": "phone_2", "action": "idle", "time": 0.1048207316385425}
{"timestamp": 1759790081.84691, "time": 0.09211773163821374, "action": "transmit_end", "id": "all_gather_reduce_from_model_parallel_region_phone_2_phone_1_0", "internal_id": 1362, "duration": 0.07533652252252253}
{"timestamp": 1759790081.846913, "device": "phone_1", "action": "running", "time": 0.09211773163821374}
{"timestamp": 1759790081.89344, "time": 0.13863189764794254, "action": "transmit_start", "id": "all_gather_reduce_from_model_parallel_region_phone_1_phone_2_0", "internal_id": 1365, "size": 8286208.0}
{"timestamp": 1759790081.8934531, "device": "phone_1", "action": "idle", "time": 0.1386457726371783}
{"timestamp": 1759790081.893468, "device": "phone_2", "action": "running", "time": 0.1048207316385425}
"""


def parse_src_dst(name: str) -> tuple[Optional[str], Optional[str]]:
    if not name:
        return None, None
    parts = name.split("_")
    try:
        idx = None
        for i in range(len(parts) - 1, -1, -1):
            if parts[i].isdigit():
                idx = i
                break
        if idx is None:
            return None, None

        def pop_name_num(j):
            if j - 1 >= 0 and parts[j - 1].isdigit() and j - 2 >= 0:
                return f"{parts[j - 2]}_{parts[j - 1]}", j - 2
            return parts[j - 1], j - 1

        dst, j = pop_name_num(idx)
        src, _ = pop_name_num(j)
        return src, dst
    except Exception:
        return None, None


def load_jsonl_or_sample(path: str) -> List[Dict[str, Any]]:
    events = []
    if os.path.exists(path):
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    events.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    if not events:
        events = [json.loads(line) for line in default_sample.strip().splitlines()]
    return events


events = load_jsonl_or_sample("../profile_out/event_log.jsonl")


starts: Dict[int, Dict[str, Any]] = {}
ends: Dict[int, Dict[str, Any]] = {}
meta: Dict[int, Dict[str, Any]] = {}

for ev in events:
    if not isinstance(ev, dict):
        continue
    action = ev.get("action", "")
    if not action.startswith("transmit_"):
        continue

    iid = ev.get("internal_id")
    if iid is None:
        continue

    if iid not in meta:
        meta[iid] = {}
    for k in ("id", "size"):
        if k in ev and k not in meta[iid]:
            meta[iid][k] = ev[k]

    if action == "transmit_start":
        starts[iid] = ev
    elif action == "transmit_end":
        ends[iid] = ev

rows = []
for iid, m in meta.items():
    s = starts.get(iid)
    e = ends.get(iid)

    name = m.get("id", f"internal_{iid}")
    size = m.get("size")
    src, dst = parse_src_dst(name)

    start_t = None
    end_t = None
    duration = None

    if s and "time" in s:
        start_t = float(s["time"])
    if e and "time" in e:
        end_t = float(e["time"])

    if s and "duration" in s:
        duration = float(s["duration"])
    if e and "duration" in e:
        duration = float(e["duration"])

    if start_t is not None and end_t is None and duration is not None:
        end_t = start_t + duration
    if end_t is not None and start_t is None and duration is not None:
        start_t = end_t - duration

    if (start_t is None or end_t is None) and e and ("time" in e) and ("duration" in e):
        end_t = float(e["time"])
        start_t = end_t - float(e["duration"])

    if start_t is None or end_t is None:
        continue

    rows.append(
        {
            "internal_id": iid,
            "name": name,
            "start": start_t,
            "end": end_t,
            "duration_s": end_t - start_t,
            "size_bytes": float(size) if size is not None else None,
            "src": src,
            "dst": dst,
        }
    )

df = pd.DataFrame(rows).sort_values(by=["start", "end", "internal_id"]).reset_index(drop=True)
df["internal_id_str"] = df["internal_id"].astype(str)
df["start"] = pd.to_numeric(df["start"], errors="coerce")
df["end"] = pd.to_numeric(df["end"], errors="coerce")

if df.empty:
    print("No intervals could be built from transmit events (need transmit_end with duration, or start+end pairs).")
else:
    fig = go.Figure()

    for name, dfg in df.groupby("name"):
        fig.add_bar(
            orientation="h",
            y=dfg["internal_id_str"],
            x=dfg["duration_s"],      # bar width = duration
            base=dfg["start"],        # bar start position
            name=name,
            hovertext=dfg.apply(
                lambda r: f"{r['name']}<br>ID:{r['internal_id']}<br>"
                        f"start:{r['start']:.6f}s <br>"
                        f"end:{r['end']:.6f}s<br>"
                        f"dur:{r['duration_s']:.6f}s <br>"
                        f"[src:{r['src']}] - [dst:{r['dst']}]<br>"
                        f"size: {r['size_bytes'] / 1024:.2f} KB" if r['size_bytes'] is not None else "size: N/A",
                axis=1,
            ),
            hoverinfo="text",
        )

    fig.update_yaxes(autorange="reversed", title="Internal ID")
    fig.update_xaxes(title="Simulation Time (s)", type="linear", rangeslider_visible=False)
    fig.update_layout(barmode="overlay", hovermode="closest", legend_title_text="Role (id)")
    display(fig)

In [128]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import re

from scipy.optimize import curve_fit

def size_to_bytes(s: str) -> float:
    units = {"B": 1, "KB": 1024, "MB": 1024*1024, "GB": 1024*1024*1024}
    m = re.match(r"(\d+(?:\.\d+)?)([KMG]?B)", s)
    if not m:
        return None
    val, unit = m.groups()
    return float(val) * units[unit]

def bytes_to_size(b: int) -> str:
    if b >= 1024**3:
        return f"{b / 1024**3:.2f} GB"
    elif b >= 1024**2:
        return f"{b / 1024**2:.2f} MB"
    elif b >= 1024:
        return f"{b / 1024:.2f} KB"
    else:
        return f"{b} B"

def plot_transfer_time_vs_size(df: pd.DataFrame, model, params):
    x = df["size_bytes"].values
    y = df["time_s"].values

    # Create interactive plot
    fig = go.Figure()

    # Scatter plot for measured data
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='markers',
        name='Measured',
        marker=dict(color='blue')
    ))

    # Line plot for fitted curve
    x_fit = np.logspace(np.log10(min(x)), np.log10(max(x)), 100)
    y_fit = model(x_fit, *params)
    fig.add_trace(go.Scatter(
        x=x_fit,
        y=y_fit,
        mode='lines',
        name='Fitted curve',
        line=dict(color='red')
    ))

    # Update layout
    fig.update_layout(
        title="Transfer Time vs Message Size",
        xaxis=dict(
            title="Message Size (bytes)",
            type="log"
        ),
        yaxis=dict(
            title="Transfer Time (s)",
            type="log"
        ),
        hovermode="closest"
    )

    fig.show()

In [129]:
df_jetson_2 = pd.read_json("../results/network_jetson_2.json")
# Normalize based on number of devices (2 devices)
df_jetson_2["size_bytes"] = df_jetson_2["size_bytes"].map(lambda x: x * 2)

df_jetson_4 = pd.read_json("../results/network_jetson_4.json")
# Normalize based on number of devices (4 devices)
df_jetson_4["size_bytes"] = df_jetson_4["size_bytes"].map(lambda x: x * 4)

# Normalize the communication and transmission parameters based on device count
# Scale bandwidth with device_count because each device sends/receives that much more data in total.
# I.e. a single device transfers 1 chunk to (device_count - 1) other devices, repeated by device_count devices.
# So total bandwidth usage scales with device_count * (device_count - 1).
df_jetson_2.loc[:, "size_bytes"] = df_jetson_2["size_bytes"] * (2-1)
df_jetson_4.loc[:, "size_bytes"] = df_jetson_4["size_bytes"] * (4-1)
df_jetson_2["transfer_count"] = 2-1
df_jetson_4["transfer_count"] = 4-1

# Sanity check
plot_transfer_time_vs_size(df=df_jetson_2, model=lambda x, a: a * x, params=[0.0001])
plot_transfer_time_vs_size(df=df_jetson_4, model=lambda x, a: a * x, params=[0.0001])

In [130]:
def linear_model(df: pd.DataFrame):
    x_size = df["size_bytes"].values
    x_count = df["transfer_count"].values
    y = df["time_s"].values

    # Define model: latency * count + size / bandwidth
    def model(size, alpha, beta):
        return alpha * 1 + beta * size

    # Fit curve
    params, _ = curve_fit(model, x_size, y)
    alpha, beta = params
    print(f"Latency per transfer (alpha): {alpha:.6f} s/transfer")
    print(f"Effective 1/bandwidth (beta): {beta:.2e} s/byte")
    print(f"Bandwidth ~ {bytes_to_size(1/beta)}/s")

    return model, params

(model, params) = linear_model(df=df_jetson_2)
plot_transfer_time_vs_size(df=df_jetson_2, model=model, params=params)

(model, params) = linear_model(df=df_jetson_4)
plot_transfer_time_vs_size(df=df_jetson_4, model=model, params=params)

Latency per transfer (alpha): 0.004651 s/transfer
Effective 1/bandwidth (beta): 8.68e-09 s/byte
Bandwidth ~ 109.86 MB/s


Latency per transfer (alpha): 0.019490 s/transfer
Effective 1/bandwidth (beta): 2.87e-09 s/byte
Bandwidth ~ 332.80 MB/s


In [131]:
def weighted_linear_model(df):
    x = df["size_bytes"].to_numpy(dtype=float)
    y = df["time_s"].to_numpy(dtype=float)

    # Latency + size/bandwidth
    def model(size, alpha, beta):
        return alpha + beta * size

    # --- Weighted least squares: equalize proportional errors ---
    # This makes a 1→2s miss cost the same as 100→200s.
    eps = 1e-12
    sigma = np.maximum(np.abs(y), eps)  # weight ~ 1/sigma^2 -> residuals become (yhat - y)/y

    params, cov = curve_fit(
        model, x, y,
        sigma=sigma,
        absolute_sigma=True,   # so cov is on an absolute scale
        maxfev=20000
    )

    alpha, beta = params
    print(f"Latency (alpha): {alpha:.6f} s")
    print(f"Effective 1/bandwidth (beta): {beta:.2e} s/byte")
    print(f"Bandwidth ~ {bytes_to_size(1.0 / beta)}/s")

    return model, params

(model, params) = weighted_linear_model(df=df_jetson_2)
plot_transfer_time_vs_size(df=df_jetson_2, model=model, params=params)

(model, params) = weighted_linear_model(df=df_jetson_4)
plot_transfer_time_vs_size(df=df_jetson_4, model=model, params=params)

Latency (alpha): 0.002166 s
Effective 1/bandwidth (beta): 8.58e-09 s/byte
Bandwidth ~ 111.20 MB/s


Latency (alpha): 0.004055 s
Effective 1/bandwidth (beta): 3.05e-09 s/byte
Bandwidth ~ 312.39 MB/s


In [143]:
device_count = 4

with open(f'../results/rtp{device_count}_sim.json', 'r') as f:
    sim_rtp_data = json.load(f)

with open(f'../results/rtp{device_count}_orin.json', 'r') as f:
    orin_rtp_data = json.load(f)

sim_rtp_df = pd.DataFrame(sim_rtp_data)
orin_rtp_df = pd.DataFrame(orin_rtp_data)

# Normalize the communication and transmission parameters based on device count
# Scale bandwidth with device_count because each device sends/receives that much more data in total.
# I.e. a single device transfers 1 chunk to (device_count - 1) other devices, repeated by device_count devices.
# So total bandwidth usage scales with device_count * (device_count - 1).
orin_rtp_df.loc[:, "total_transmit_size"] = orin_rtp_df["total_transmit_size"] * (device_count-1)
sim_rtp_df.loc[:, "total_transmit_size"] = sim_rtp_df["total_transmit_size"] / (device_count)
sim_rtp_df.loc[:, "total_transmit_duration"] = sim_rtp_df["total_transmit_duration"] / (device_count)
# The overhead of transmit_count is per transfer, not per device.
orin_rtp_df.loc[:, "transmit_count"] = orin_rtp_df["transmit_count"] * (device_count - 1)


In [145]:
import pandas as pd

# Assuming sim_rtp_df and orin_rtp_df have matching rows by index
comparison_df = pd.DataFrame({
    'prompt_length': sim_rtp_df['prompt_length'],
    'max_tokens': sim_rtp_df['max_tokens'],
    'transmit_count': sim_rtp_df['transmit_count'],
    'sim_duration': sim_rtp_df['total_transmit_duration'],
    'orin_duration': orin_rtp_df['total_transmit_duration'],
    'duration_rel_error': ((sim_rtp_df['total_transmit_duration'] - orin_rtp_df['total_transmit_duration']) / orin_rtp_df['total_transmit_duration']) * 100,  # % error
    'sim_size': sim_rtp_df['total_transmit_size'],
    'orin_size': orin_rtp_df['total_transmit_size'],
    'size_rel_error': ((sim_rtp_df['total_transmit_size'] - orin_rtp_df['total_transmit_size']) / orin_rtp_df['total_transmit_size']) * 100  # % error
})

# Format the DataFrame for display
comparison_df['sim_duration'] = comparison_df['sim_duration'].round(2)
comparison_df['orin_duration'] = comparison_df['orin_duration'].round(2)
comparison_df['duration_rel_error'] = comparison_df['duration_rel_error'].round(2).astype(str) + '%'
comparison_df['sim_size'] = comparison_df['sim_size'].apply(lambda x: bytes_to_size(int(x)) if not pd.isna(x) else 'N/A')
comparison_df['orin_size'] = comparison_df['orin_size'].apply(lambda x: bytes_to_size(int(x)) if not pd.isna(x) else 'N/A')
comparison_df['size_rel_error'] = comparison_df['size_rel_error'].round(2).astype(str) + '%'

# Create a Plotly table for prettier display
fig = go.Figure(data=[go.Table(
    header=dict(values=list(comparison_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[comparison_df[col] for col in comparison_df.columns],
               fill_color='lavender',
               align='left'))
])

fig.update_layout(title="Comparison of Simulation and Orin RTP Data")
fig.show()

In [134]:
train_df = orin_rtp_df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]

In [135]:
import numpy as np
from scipy.optimize import curve_fit
from sklearn.linear_model import HuberRegressor

import plotly.express as px
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

# Fit the model on train_df: total_transmit_duration = a * transmit_count + b * total_transmit_size
X = train_df[["transmit_count", "total_transmit_size"]].values
y = train_df["total_transmit_duration"].values

a = 0.001611990775692557
b = 7.827260712053543e-09

def model_func(x, a, b):
    return a * x[:, 0] + b * x[:, 1]

# # Use weighted least squares to reduce impact of outliers causing overshooting
# eps = 1e-12
sigma = np.maximum(np.abs(y), eps)

params, _ = curve_fit(
    model_func, X, y,
    # sigma=sigma,
    # absolute_sigma=True,
    # p0=[a, b],           # <-- use your seeds
    bounds=(0, np.inf),    # <-- optional: enforce non-negative coeffs
    maxfev=20000
)
a, b = params

# A = np.column_stack([X[:,0], X[:,1]])
# (a, b), *_ = np.linalg.lstsq(A, y, rcond=None)

# # Weighted (same sigma semantics as curve_fit)
# w = 1.0 / np.maximum(np.abs(y), 1e-12)**2
# Aw = A * np.sqrt(w)[:,None]
# yw = y * np.sqrt(w)
# (a, b), *_ = np.linalg.lstsq(Aw, yw, rcond=None)

print(f"Fitted a (latency coefficient): {a}")
print(f"Fitted b (1/bandwidth coefficient): {b}")

# Compute predictions and accuracy metrics
y_true = train_df["total_transmit_duration"].values
y_pred = a * train_df["transmit_count"] + b * train_df["total_transmit_size"]

print(f"R2 Score: {r2_score(y_true, y_pred)}")
print(f"MSE: {mean_squared_error(y_true, y_pred)}")
print(f"MAE: {mean_absolute_error(y_true, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_true, y_pred)}")

for transmit_count, group in orin_rtp_df.groupby("transmit_count"):
    sorted_group = group.sort_values("transmit_count")
    fig = px.box(sorted_group, x="transmit_count", y="total_transmit_duration", title=f"Transmit_count: {transmit_count}")

    # Add prediction curve
    predicted = a * sorted_group["transmit_count"] + b * sorted_group["total_transmit_size"]
    fig.add_trace(
        go.Scatter(x=sorted_group["transmit_count"], y=predicted, mode="lines", name="Predicted", line=dict(dash="dash"))
    )

    fig.update_layout(xaxis_title="Number of Tokens", yaxis_title="Total Transmit Duration (s)")
    fig.show()


Fitted a (latency coefficient): 0.0007232350567015374
Fitted b (1/bandwidth coefficient): 5.038396466332825e-09
R2 Score: 0.99812465522797
MSE: 0.009358488269760616
MAE: 0.08057863190943654
MAPE: 0.08679776705204127


In [136]:
# If you don't have plotly installed, uncomment:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go


def fit_model(train_df: pd.DataFrame):
    # --- Columns ---
    y_col = "total_transmit_duration"
    x_size = "total_transmit_size"
    x_count = "transmit_count"

    # --- Fit OLS: duration ~ size + count ---
    X = train_df[[x_size, x_count]].copy()
    y = train_df[y_col].copy()
    X_const = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X_const).fit()

    print(model.params)

    # --- Build a prediction grid over size × count ---
    n_side = 50  # grid resolution per axis (increase for smoother surface)
    size_lin = np.linspace(train_df[x_size].min(), train_df[x_size].max(), n_side)
    count_lin = np.linspace(train_df[x_count].min(), train_df[x_count].max(), n_side)
    S, C = np.meshgrid(size_lin, count_lin)

    grid_df = pd.DataFrame({"const": 1.0, x_size: S.ravel(), x_count: C.ravel()})

    # Mean prediction
    mean_pred = model.get_prediction(grid_df).summary_frame(alpha=0.05)  # 95%
    Z_mean = mean_pred["mean"].values.reshape(S.shape)

    # 95% Prediction Interval (for new observations)
    Z_pi_low = mean_pred["obs_ci_lower"].values.reshape(S.shape)
    Z_pi_high = mean_pred["obs_ci_upper"].values.reshape(S.shape)

    # --- Build interactive 3D figure ---
    fig = go.Figure()

    # Fitted surface (mean)
    fig.add_trace(go.Surface(x=S, y=C, z=Z_mean, name="Fitted surface (mean)", showscale=False, opacity=0.85))

    # Raw data points
    fig.add_trace(
        go.Scatter3d(
            x=train_df[x_size],
            y=train_df[x_count],
            z=train_df[y_col],
            mode="markers",
            name="Data",
            marker=dict(size=3, opacity=0.6),
        )
    )

    # Optional: lower PI surface (toggle via legend)
    fig.add_trace(
        go.Surface(x=S, y=C, z=Z_pi_low, name="Lower 95% PI", showscale=False, opacity=0.25, visible="legendonly")
    )

    # Optional: upper PI surface (toggle via legend)
    fig.add_trace(
        go.Surface(x=S, y=C, z=Z_pi_high, name="Upper 95% PI", showscale=False, opacity=0.25, visible="legendonly")
    )

    fig.update_layout(
        title="Interactive 3D: Duration ~ Size + Transmit Count",
        scene=dict(
            xaxis_title="Total Transmit Size",
            yaxis_title="Transmit Count",
            zaxis_title="Total Transmit Duration",
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)),
        ),
        legend=dict(itemsizing="constant"),
    )

    fig.show()

    # Optional: quick coefficients readout
    print(model.summary().tables[1])

    # Predictions
    y_pred = model.predict(X_const)  # X_const is your design matrix with constant
    y_true = y

    # Compute relative % error
    rel_error = (y_pred - y_true) / y_true * 100

    # Put into a table
    error_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "rel_error_%": rel_error})

    display(error_df.sort_values(by="rel_error_%"))


In [137]:
import json

with open('../results/rtp2_orin.json', 'r') as f:
    orin_rtp_2_data = json.load(f)

with open('../results/rtp4_orin.json', 'r') as f:
    orin_rtp_4_data = json.load(f)

orin_rtp_2_df = pd.DataFrame(orin_rtp_2_data)
orin_rtp_2_df = orin_rtp_2_df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
orin_rtp_2_df.loc[:, "total_transmit_size"] = orin_rtp_2_df["total_transmit_size"] * 2
orin_rtp_2_df.loc[:, "transmit_count"] = orin_rtp_2_df["transmit_count"] * 2

orin_rtp_4_df = pd.DataFrame(orin_rtp_4_data)
orin_rtp_4_df = orin_rtp_4_df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
orin_rtp_4_df.loc[:, "total_transmit_size"] = orin_rtp_4_df["total_transmit_size"] * 4
orin_rtp_4_df.loc[:, "transmit_count"] = orin_rtp_4_df["transmit_count"] * 4

fit_model(orin_rtp_2_df)
fit_model(orin_rtp_4_df)

const                 -2.964413e-01
total_transmit_size    8.309075e-09
transmit_count         1.667335e-03
dtype: float64


                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.2964      0.017    -17.457      0.000      -0.330      -0.263
total_transmit_size  8.309e-09   4.32e-11    192.394      0.000    8.22e-09    8.39e-09
transmit_count          0.0017   6.37e-06    261.603      0.000       0.002       0.002


,y_true,y_pred,rel_error_%
0,0.329671,0.231768,-29.697075
2,0.314214,0.231768,-26.238746
5,0.289978,0.231768,-20.073866
9,0.276641,0.231768,-16.220649
3,0.275463,0.231768,-15.862285
...,...,...,...
71,1.236163,1.425540,15.319756
65,0.925467,1.072271,15.862683
78,1.227066,1.425540,16.174707
70,1.216773,1.425540,17.157498


const                 -1.171262e-01
total_transmit_size    5.183591e-09
transmit_count         5.550796e-04
dtype: float64


                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.1171      0.010    -11.633      0.000      -0.137      -0.097
total_transmit_size  5.184e-09   2.22e-11    233.276      0.000    5.14e-09    5.23e-09
transmit_count          0.0006   2.03e-06    272.927      0.000       0.001       0.001


,y_true,y_pred,rel_error_%
0,0.173220,0.150670,-13.018529
108,4.739083,4.316004,-8.927452
21,0.486742,0.455655,-6.386685
1,0.158598,0.150670,-4.999321
36,0.366557,0.349019,-4.784680
...,...,...,...
11,1.277345,1.345016,5.297832
19,0.204788,0.216786,5.858860
7,0.665908,0.708031,6.325756
10,1.263330,1.345016,6.465898


In [138]:
with open('../results/rtp2_sim.json', 'r') as f:
    sim_rtp_2_data = json.load(f)

with open('../results/rtp4_sim.json', 'r') as f:
    sim_rtp_4_data = json.load(f)

sim_rtp_2_df = pd.DataFrame(sim_rtp_2_data)
sim_rtp_2_df = sim_rtp_2_df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
sim_rtp_2_df.loc[:, "total_transmit_size"] = sim_rtp_2_df["total_transmit_size"] * 2
sim_rtp_2_df.loc[:, "transmit_count"] = sim_rtp_2_df["transmit_count"] * 2

sim_rtp_4_df = pd.DataFrame(sim_rtp_4_data)
sim_rtp_4_df = sim_rtp_4_df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
sim_rtp_4_df.loc[:, "total_transmit_size"] = sim_rtp_4_df["total_transmit_size"] * 4
sim_rtp_4_df.loc[:, "transmit_count"] = sim_rtp_4_df["transmit_count"] * 4

fit_model(sim_rtp_2_df)
fit_model(sim_rtp_4_df)

const                  3.177945e-02
total_transmit_size    7.829417e-09
transmit_count         1.487806e-03
dtype: float64


                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.0318      0.006      5.440      0.000       0.020       0.043
total_transmit_size  7.829e-09   7.43e-12   1053.950      0.000    7.81e-09    7.84e-09
transmit_count          0.0015    1.1e-06   1357.123      0.000       0.001       0.001


,y_true,y_pred,rel_error_%
124,9.018392,8.935388,-0.920381
121,9.017625,8.935388,-0.911960
120,9.016187,8.935388,-0.896153
135,9.653213,9.567163,-0.891416
51,14.405312,14.282818,-0.850344
...,...,...,...
5,0.982749,1.015556,3.338252
4,0.982713,1.015556,3.342113
8,0.981408,1.015556,3.479485
1,0.981329,1.015556,3.487856


const                 -2.809095e-01
total_transmit_size    2.005890e-09
transmit_count         1.506476e-03
dtype: float64


                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.2809      0.253     -1.108      0.270      -0.783       0.221
total_transmit_size  2.006e-09   4.66e-11     43.018      0.000    1.91e-09     2.1e-09
transmit_count          0.0015   4.27e-06    352.988      0.000       0.001       0.002


,y_true,y_pred,rel_error_%
74,8.871019,7.689750,-13.316044
73,8.827408,7.689750,-12.887790
76,17.246020,15.123025,-12.310057
80,27.214178,25.034058,-8.010973
60,24.252965,22.577896,-6.906655
...,...,...,...
1,2.916470,3.084447,5.759594
5,9.936874,10.517722,5.845371
40,10.744261,11.438782,6.464109
41,10.704885,11.438782,6.855722
